# Why the manifold kernel doesn't beat a Euclidean Matérn

Two parts, both scored on the **real fold-2 split** used by the actual deployed models
(`data/splits/fold_2.json`): trained on 6 wild-type brains, evaluated on 2 held-out ones
(Male2, Male3) that never contribute to any statistic a model is fit against. Earlier versions of
this notebook pooled all 8 brains together before splitting — a materially easier, same-data
interpolation task that overstated every number here. Rebuilt from scratch on genuine cross-subject
generalization; see `manifold/analysis/build_fold2_pools.py`.

**Part 1 — lengthscale sweep.** Which distance metric best explains the data, how big the
anatomical-boundary "prize" actually is, a lengthscale sweep locking one Euclidean and one manifold
kernel by genuine held-out predictive correlation, whether the manifold's decay rate is what's costing
it, and where its spectral weight actually lives.

**Part 2 — actual trained models.** The real, deployed per-lipid GP checkpoints, each evaluated on the
lipid it was trained on — the same fold-2 split, but the real training pipeline instead of this
notebook's closed-form stand-in.

**Part 3 — three deployed kernels in depth.** Real learned hyperparameters, the same four checks, and
the weakness of free per-mode weights.

---
# Part 1 — Lengthscale sweep

**The split.** TRAIN pool: 3,000 anchor cells (0.2mm, >=20 voxels/lipid, same grid as before) pooled
from only the 6 fold-2 training brains. TEST pool: 3,000 anchor cells pooled from only the 2 held-out
brains (Male2, Male3), z-scored using the **TRAIN pool's own mean/std** (never the test brains' own
statistics — that would leak test information into normalization). All 178 lipids. Both pools are
built once by `manifold/analysis/build_fold2_pools.py` and cached; this notebook only loads them.

**Why not the old pooled-then-random-CV design?** Pooling all 8 brains together first (as earlier
versions of this notebook did) bakes the two "held-out" brains' own data into every training point via
the averaging step, then a random split of that blend answers "how well can a kernel interpolate an
already-blended pattern" — not "does this transfer to a brain the model never saw." That gap turned out
to be large (see Part 2): most of a kernel's apparent performance on the old pooled design was optimism
from data leakage, not real predictive power.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist, squareform, pdist
from scipy.stats import wilcoxon, spearmanr

REPO = Path("/home/casap/mlibra_git")
for _p in (REPO, REPO / "maldi", REPO / "manifold"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
from utils import crop_or_stride_volume, reference_ccf_from_subvolume
from manifold_gp.utils.anatomical_knn import (
    labels_for_nodes_from_sub_atlas, dissolve_root_labels)

# ============================ WHICH FOLD =====================================
# Everything below is derived from FOLD, so switching folds is a one-line edit.
# The train/test brains come from the same split file the training runs use
# (maldi/data/splits/fold_<N>.json) rather than being restated here -- for fold 2
# that yields exactly the 6 + 2 brains the original hard-coded lists had.
FOLD = 2

SPLIT_FILE = REPO / "maldi" / "data" / "splits" / f"fold_{FOLD}.json"
_split = json.loads(SPLIT_FILE.read_text())
TRAIN_SAMPLES = [f[2] for f in _split["train"]]
TEST_SAMPLES = [f[2] for f in _split["test"]]

RUN_PREFIX = f"FOLD-{FOLD}-"                  # trained run dirs for this fold
SWEEP_CACHE = REPO / "manifold" / "analysis"  # fold<N>_*.npz sweep results
POOL_CACHE = Path.cwd() / f"pools_fold{FOLD}.npz"
print(f"fold {FOLD}: train {TRAIN_SAMPLES}  |  held out {TEST_SAMPLES}")

# ===================== anchor pools + region geometry ========================
# Self-contained: this rebuilds what analysis/ used to hand over as opaque
# caches (fold<N>_{train,test}_full.npz and laplacian_distance_results.npz),
# from the reference volume, the Allen annotation and the MALDI parquet.
# Verified bit-for-bit against those caches on fold 2: anch_node/anch_mm
# identical, Z_train and Z_test max|diff| = 0, labels identical, border
# distance max|diff| 1.1e-06.
DATA = Path("/home/casap/mlibra/mlibra_data")
REFERENCE_FILE, ANNOTATION_FILE = DATA / "reference_image.npy", DATA / "level_15annot.npy"
MALDI_FILE = DATA / "maindata_minimal.parquet"
AVAILABLE_LIPIDS_FILE = DATA / "maindata_minimal_available_lipids.npy"

def _cache_matches(path):
    """Reject a cache built with a different channel set (e.g. the old 178)."""
    try:
        n = len(np.load(path, allow_pickle=True)["lipids"])
    except Exception:
        return False
    want = len(np.load(AVAILABLE_LIPIDS_FILE, allow_pickle=True))
    if n != want:
        print(f"  {path.name}: {n} channels, expected {want} -- rebuilding")
        return False
    return True


STRIDE, THRESHOLD = 4, 5
COORD_COLS = ("xccf", "yccf", "zccf")
MIN_VOXELS, POOL_SIZE, SNAP_MM = 20, 3000, 1.0
REBUILD_POOLS = False

_ref, _ann = np.load(REFERENCE_FILE), np.load(ANNOTATION_FILE)
SUB, SUB_ATLAS, _off, _vs = crop_or_stride_volume(_ref, _ann, STRIDE)
NODES = reference_ccf_from_subvolume(SUB, _off, _vs, THRESHOLD).astype(np.float64)
NODE_IDX = np.argwhere(SUB > THRESHOLD).astype(np.int32)
_, CELL_OF_NODE = np.unique(NODE_IDX // 2, axis=0, return_inverse=True)   # 0.2 mm cells
NCELL = int(CELL_OF_NODE.max()) + 1

# region label per node, with the level_15 'root' catch-all dissolved into the
# nearest real region (no majority-vote denoise -- the cached array had none),
# and distance to the nearest node of a DIFFERENT region.
LABELS = dissolve_root_labels(
    labels_for_nodes_from_sub_atlas(SUB, SUB_ATLAS, THRESHOLD), NODES.astype(np.float32))
D_INTER = np.empty(len(NODES))
for _lab in np.unique(LABELS):
    _t = LABELS == _lab
    D_INTER[_t] = cKDTree(NODES[LABELS != _lab]).query(NODES[_t], k=1, workers=-1)[0]


def _snap(coords_mm, node_ccf, max_mm=SNAP_MM):
    """Nearest template node per MALDI voxel; drops points farther than max_mm."""
    n = node_ccf.shape[0]
    pts = coords_mm.astype(np.float32)
    finite = np.isfinite(pts).all(1)
    idx = np.zeros(len(pts), np.int64); dist = np.full(len(pts), np.inf)
    if finite.any():
        d, i = cKDTree(node_ccf).query(pts[finite], k=1); dist[finite] = d; idx[finite] = i
    oob = (~finite) | ~np.isfinite(dist) | (idx >= n) | (idx < 0)
    return (np.where(oob, 0, idx).astype(np.int64),
            np.isfinite(dist) & (dist <= max_mm) & ~oob)


def _accumulate(samples, node_mm, lipids, tag):
    """Mean lipid value per 0.2 mm cell, and the cells with full coverage."""
    ssum = np.zeros((NCELL, len(lipids))); cnt = np.zeros((NCELL, len(lipids)), np.int32)
    for smp in samples:
        df = pd.read_parquet(MALDI_FILE, columns=[*COORD_COLS, *lipids],
                             filters=[("Sample", "==", smp)])
        ni, valid = _snap(df[list(COORD_COLS)].to_numpy(np.float32), node_mm)
        vals = df[lipids].to_numpy(np.float32)[valid]; ci = CELL_OF_NODE[ni[valid]]
        fin = np.isfinite(vals)
        np.add.at(ssum, ci, np.where(fin, vals, 0.0))
        np.add.at(cnt, ci, fin.astype(np.int32))
        print(f"  [{tag}] {smp:<15} {int(valid.sum()):>9,} voxels", flush=True)
        del df, vals, fin, ci
    with np.errstate(invalid="ignore", divide="ignore"):
        Y = np.where(cnt > 0, ssum / np.maximum(cnt, 1), np.nan)
    return Y, np.where((cnt >= MIN_VOXELS).all(1))[0]


def _rep_nodes(cells, node_mm):
    """One representative node per cell: the one nearest that cell's centroid.

    Recomputed per eligible-cell set -- reusing the TRAIN reps for the TEST
    cells would silently default any test-only cell to node 0.
    """
    rep = np.zeros(NCELL, np.int64); order = np.argsort(CELL_OF_NODE, kind="stable")
    st = np.searchsorted(CELL_OF_NODE[order], np.arange(NCELL))
    en = np.searchsorted(CELL_OF_NODE[order], np.arange(NCELL), side="right")
    for k in cells:
        mem = order[st[k]:en[k]]
        rep[k] = mem[np.argmin(((node_mm[mem] - node_mm[mem].mean(0)) ** 2).sum(1))]
    return rep


if POOL_CACHE.exists() and _cache_matches(POOL_CACHE) and not REBUILD_POOLS:
    _P = dict(np.load(POOL_CACHE, allow_pickle=True))
    print(f"anchor pools: reusing {POOL_CACHE.name}")
else:
    import pyarrow as pa, pyarrow.parquet as pq
    _nm = NODES.astype(np.float32)          # snap in float32, as the pools were built
    # Take the channel list from the available-lipids file -- the same one the
    # training runs use -- NOT "every float column". The parquet has 178 float
    # columns, but 5 of them (x, y, x_index, y_index, z_index) are coordinates,
    # and build_fold2_pools.py pooled them as if they were lipids. They are
    # perfectly smooth in space, so they inflate the empirical covariance by
    # +0.009 at 0.3 mm rising to +0.015 at 2 mm -- i.e. they prop up the very
    # correlogram tail the decay-rate analysis is about. 173 real lipids.
    _avail = {str(x) for x in np.load(AVAILABLE_LIPIDS_FILE, allow_pickle=True)}
    _schema = pq.read_schema(MALDI_FILE)
    _lip = [c for c in _schema.names if c in _avail]
    assert len(_lip) == len(_avail), f"{len(_avail) - len(_lip)} listed lipids missing from the parquet"
    print(f"building fold-{FOLD} anchor pools from {len(_lip)} lipids (~2 min) ...", flush=True)
    _Ytr, _ctr = _accumulate(TRAIN_SAMPLES, _nm, _lip, "TRAIN")
    _mu, _sd = np.nanmean(_Ytr[_ctr], 0), np.nanstd(_Ytr[_ctr], 0) + 1e-8
    _ptr = np.sort(np.random.default_rng(0).choice(_ctr, min(POOL_SIZE, len(_ctr)), replace=False))
    _ntr = _rep_nodes(_ctr, _nm)[_ptr]
    # held-out brains are z-scored with the TRAIN pool's statistics, never their
    # own -- which is why both pools are built even when only one is wanted
    _Yte, _cte = _accumulate(TEST_SAMPLES, _nm, _lip, "TEST")
    _pte = np.sort(np.random.default_rng(1).choice(_cte, min(POOL_SIZE, len(_cte)), replace=False))
    _nte = _rep_nodes(_cte, _nm)[_pte]
    assert len(np.unique(_nte)) == len(_pte), "duplicate anchor nodes in the TEST pool"
    _P = dict(anch_node_tr=_ntr, anch_mm_tr=_nm[_ntr].astype(np.float64),
              Z_train=((_Ytr[_ptr] - _mu) / _sd).astype(np.float64),
              anch_node_te=_nte, anch_mm_te=_nm[_nte].astype(np.float64),
              Z_test=((_Yte[_pte] - _mu) / _sd).astype(np.float64),
              lipids=np.array(_lip))
    np.savez(POOL_CACHE, **_P)
    print(f"anchor pools -> {POOL_CACHE.name}")

LIP = [str(x) for x in _P["lipids"]]
L = len(LIP)
anch_node_tr, anch_mm_tr, Z_tr = (_P["anch_node_tr"], _P["anch_mm_tr"].astype(np.float64),
                                  _P["Z_train"].astype(np.float64))
anch_node_te, anch_mm_te, Z_te = (_P["anch_node_te"], _P["anch_mm_te"].astype(np.float64),
                                  _P["Z_test"].astype(np.float64))
NTR, NTE = len(anch_node_tr), len(anch_node_te)
REGION_tr, REGION_te = LABELS[anch_node_tr], LABELS[anch_node_te]
DB_tr, DB_te = D_INTER[anch_node_tr], D_INTER[anch_node_te]

EIGDIR = Path("/home/casap/mlibra/output/eigenvectors/eigvecs")
GRAPH_FILES = {
    "faiss":                 (EIGDIR / "bw=0.1_graph=bbox=None_k=15_method=faiss_stride=4_template=reference_thresh=5_modes=1000_norm=randomwalk.eigpairs.npz", 1000),
    "atlas x50":             (EIGDIR / "bw=0.1_graph=bbox=None_k=15_method=faiss_atlas_weighted_nlist=729_stride=4_template=reference_thresh=5_weighting=atlas_x50_rootdissolve_modes=1300_norm=randomwalk.eigpairs.npz", 1300),
    "atlas x50 + prune .95": (EIGDIR / "bw=0.1_graph=bbox=None_denoise=3_k=15_method=faiss_atlas_weighted_nlist=729_prune=0.95_stride=4_template=reference_thresh=5_weighting=atlas_x50_rootdissolve_modes=1300_norm=randomwalk.eigpairs.npz", 1300),
}
GRAPHS = list(GRAPH_FILES)
EIGVAL, VEC_TR, VEC_TE = {}, {}, {}
for name, (f, nmodes) in GRAPH_FILES.items():
    d = np.load(f)
    EIGVAL[name] = d["eigval"].astype(np.float64)
    VEC_TR[name] = d["eigvec"][anch_node_tr].astype(np.float64)
    VEC_TE[name] = d["eigvec"][anch_node_te].astype(np.float64)

T_HEAT = float(1.0 / EIGVAL["faiss"][200])

def diffusion(graph, side, n_modes=None):
    """side='tr' or 'te'. Rows embedded so ||E_i - E_j|| is the diffusion distance."""
    lam = EIGVAL[graph][:n_modes]
    vec = (VEC_TR if side == "tr" else VEC_TE)[graph][:, :len(lam)]
    return vec * np.exp(-T_HEAT * lam)[None, :]

D = {
    "euclidean":              {"tr": cdist(anch_mm_tr, anch_mm_tr), "te": cdist(anch_mm_te, anch_mm_te)},
}
for g in GRAPHS:
    D[f"laplacian · {g}"] = {"tr": squareform(pdist(diffusion(g, "tr"))).astype(np.float64),
                              "te": squareform(pdist(diffusion(g, "te"))).astype(np.float64)}
METRICS = list(D)

COL = {"euclidean": "#6a7079",
       "laplacian · faiss": "#4a3aa7", "laplacian · atlas x50": "#e34948",
       "laplacian · atlas x50 + prune .95": "#eb6834", "tissue identity": "#008300"}
INK2, GRID = "#52514e", "#e2e1dd"
mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": .6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
    "axes.titlesize": 10, "legend.frameon": False})

print(f"TRAIN pool: {NTR:,} anchors (6 brains)   TEST pool: {NTE:,} anchors (Male2+Male3, held out)")
print(f"lipids: {L}")
print(f"metrics: {METRICS}")

## Which distance explains the lipid data?

For lipid $\ell$ and a pair of cells $(i,j)$, the product $p_{ij} = z_\ell(i)\,z_\ell(j)$ is a
one-sample estimate of their correlation. **Fit** a quantile-binned mean $\bar p(d)$ on **TRAIN pool**
pairs only, then **score** its $R^2$ predicting $p_{ij}$ on **TEST pool** pairs — genuine cross-subject
generalization, not a random split of the same blended data. Aggregate the per-lipid $R^2$ (median
across all 178 lipids) and bootstrap over random TEST-anchor subsets for a confidence interval.

> $\operatorname{Var}(p_{ij}) \approx 1$ regardless of the true structure (a single product is a
> 1-degree-of-freedom estimate of a correlation) — so $R^2$ is small by construction. **Read the
> ratios, not the levels.**

In [ ]:
MAX_MM = 2.0
N_BOOT, N_SUB = 16, 2500

# ---- bootstrap over held-out TEST anchors (16 draws x 2500-anchor subsamples,
# quantile-bin R^2 fit on TRAIN / scored on TEST) -- computed once and cached; loaded here directly.
BOOT_CACHE = SWEEP_CACHE / f"fold{FOLD}_which_distance_cache.npz"
if not BOOT_CACHE.exists():
    raise FileNotFoundError(f"{BOOT_CACHE} missing -- rebuild with:  python manifold/analysis/build_which_distance_cache.py")
_c = np.load(BOOT_CACHE, allow_pickle=True)
_nlip_cache = _c["boot_0"].shape[1]   # 178: this cache predates the lipid-list fix
boot = {m: _c[f"boot_{i}"] for i, m in enumerate(METRICS)}

print(f"{N_BOOT} draws x {N_SUB} TEST anchors, fit on {NTR} TRAIN anchors\n")

def stat(m):
    per_draw = np.nanmedian(boot[m], axis=1)
    return per_draw.mean(), np.percentile(per_draw, [2.5, 97.5])

base = stat("euclidean")[0]
print(f"{'metric':<34}{'test R² [95% CI]':>26}{'vs Euclid':>11}{'draws won':>11}{'lipids won':>12}")
for m in METRICS:
    mu, ci = stat(m)
    dwins = int((np.nanmedian(boot[m], 1) > np.nanmedian(boot["euclidean"], 1)).sum())
    a = np.nanmedian(boot[m], 0); e = np.nanmedian(boot["euclidean"], 0)
    lwins = int((a > e).sum())
    print(f"{m:<34}{mu:>20.5f} [{ci[0]:.4f},{ci[1]:.4f}]{mu/base:>10.2f}x"
          f"{f'{dwins}/{N_BOOT}':>11}{f'{lwins}/{L}':>12}")

print(f"\npaired Wilcoxon across the {_nlip_cache} cached channels, vs Euclidean:")
for m in METRICS[1:]:
    a = np.nanmedian(boot[m], 0); e = np.nanmedian(boot["euclidean"], 0)
    ok = np.isfinite(a) & np.isfinite(e)
    print(f"   {m:<34}p = {wilcoxon(a[ok], e[ok], alternative='greater').pvalue:.1e}")

**Scored properly (bin-level, not per raw pair), Euclidean wins here too.** The version of this
check used earlier in this notebook scored each model against *individual* pairwise products
`z(i)·z(j)` — a 1-sample estimate of a correlation with `Var≈1` regardless of the true structure (see
the discussion elsewhere in this notebook), which is dominated by noise no model can reduce. Scoring
instead at the level the correlogram is actually plotted at — TRAIN-fit bin means, weighted R² against
that same TEST draw's own bin means — removes that noise floor and **reverses the earlier result**:
Euclidean wins every one of the 16 bootstrap draws, on every graph (`faiss`, `atlas x50`, pruned).
This is also the version consistent with the rest of the notebook (the lengthscale-lock and real-kernel
sections both already showed Euclidean winning on genuine held-out correlation) — the old per-pair
result was the one outlier in the other direction, and it was an artifact of the scoring metric, not a
real advantage for the graph kernel.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

ax = axes[0]
idx0 = np.random.default_rng(0).choice(NTE, 2500, replace=False)
I0, J0 = np.triu_indices(len(idx0), 1)
keep0 = D["euclidean"]["te"][np.ix_(idx0, idx0)][I0, J0] <= MAX_MM
I0, J0 = I0[keep0], J0[keep0]
S0 = np.nanmean(Z_te[idx0][I0] * Z_te[idx0][J0], axis=1)
N_PCTILE = 12
pctile_ctrs = 100 * (np.arange(N_PCTILE) + 0.5) / N_PCTILE
for m in ["euclidean", "laplacian · atlas x50"]:
    d0 = D[m]["te"][np.ix_(idx0, idx0)][I0, J0]
    e = np.quantile(d0, np.linspace(0, 1, N_PCTILE + 1))
    cy = [S0[(d0 >= e[k]) & (d0 < e[k+1])].mean() for k in range(N_PCTILE)]
    ax.plot(pctile_ctrs, cy, color=COL[m], lw=2, marker="o", ms=3, label=m.split(" · ")[0])
ax.axhline(0, color=INK2, lw=.8, ls=":")
ax.set_xlabel("distance percentile (within each metric's own ≤2mm-euclidean pair pool)")
ax.set_ylabel("mean lipid correlation")
ax.set_title("(a) Correlogram — TEST pool (Male2+Male3), held out")
ax.legend(fontsize=8)

ax = axes[1]
xx = np.arange(len(METRICS))
mus = [stat(m)[0] for m in METRICS]
cis = np.array([stat(m)[1] for m in METRICS]).T
ax.bar(xx, mus, .6, yerr=np.abs(cis - np.array(mus)), error_kw=dict(lw=1, ecolor=INK2, capsize=3),
       color=[COL[m] for m in METRICS], edgecolor="white", linewidth=.8)
ax.set_xticks(xx); ax.set_xticklabels([m.replace(" · ", "\n") for m in METRICS], fontsize=7, rotation=20, ha="right")
ax.set_ylabel(f"test R² (median over {_nlip_cache} channels)")
ax.set_title(f"(b) Train on 6 brains, test on 2 held out\n({N_BOOT} bootstrap draws, 95% CI)")

fig.suptitle("Which distance explains the lipid data — real cross-subject generalization", fontsize=12, y=1.04)
fig.tight_layout()
plt.show()

## The border prize — how big is it, with no kernel fitted?

A stationary kernel can only ever use distance. The most it could ever get right is the binned mean
$\bar C(d)$ — but the data splits into $C_{\rm same}(d)$ and $C_{\rm cross}(d)$. That gap, weighted by
how many pairs actually cross a border, is the upper bound on what any border-aware kernel could add —
no model, no fitting. Computed on the **TRAIN pool** (what a model is actually fit against), with a
check that the same pattern holds in the **TEST pool** (genuinely unseen brains) too.

In [ ]:
EDG = np.array([0.2, 0.4, 0.6, 0.8, 1.0, 1.3, 1.6, 2.0])

def border_report(anch_mm, Z, REGION, DB, label):
    print(f"=== {label} ===")
    print(f"   {100*(DB<=0.2).mean():.1f}% of anchors within 0.2mm of another region")
    I, J = np.triu_indices(len(anch_mm), 1)
    dd = np.linalg.norm(anch_mm[I] - anch_mm[J], axis=1)
    sel = dd <= 2.0
    I, J, dd = I[sel], J[sel], dd[sel]
    S_emp = (Z[I] * Z[J]).mean(1)
    XR = REGION[I] != REGION[J]
    print(f"   {'d (mm)':<11}{'n same':>9}{'n cross':>9}{'% cross':>9}{'C_same':>9}{'C_cross':>9}{'gap':>8}")
    for b in range(len(EDG) - 1):
        m = (dd >= EDG[b]) & (dd < EDG[b+1])
        ns, nc = int((m & ~XR).sum()), int((m & XR).sum())
        cs, cc = S_emp[m & ~XR].mean(), S_emp[m & XR].mean()
        print(f"   {f'{EDG[b]}-{EDG[b+1]}':<11}{ns:>9,}{nc:>9,}{100*nc/(ns+nc):>8.0f}%"
              f"{cs:>9.3f}{cc:>9.3f}{cs-cc:>8.3f}")
    NB = 24
    q = np.quantile(dd, np.linspace(0, 1, NB + 1)); q[-1] += 1e-9
    bin_ = np.clip(np.digitize(dd, q) - 1, 0, NB - 1)
    C_dist = np.zeros(len(S_emp)); C_both = np.zeros(len(S_emp))
    for k in range(NB):
        m = bin_ == k
        C_dist[m] = S_emp[m].mean()
        for grp in (False, True):
            mm = m & (XR == grp)
            if mm.sum() > 5: C_both[mm] = S_emp[mm].mean()
    SS_tot = ((S_emp - S_emp.mean()) ** 2).sum()
    SS_dist = ((S_emp - C_dist) ** 2).sum(); SS_both = ((S_emp - C_both) ** 2).sum()
    prize = (SS_dist - SS_both) / SS_tot
    print(f"   distance alone: {100*(1-SS_dist/SS_tot):.2f}%   distance+border: {100*(1-SS_both/SS_tot):.2f}%"
          f"   >>> border adds: {100*prize:.2f}%\n")
    return prize

prize_tr = border_report(anch_mm_tr, Z_tr, REGION_tr, DB_tr, "TRAIN pool (6 brains) — what a kernel is fit against")
prize_te = border_report(anch_mm_te, Z_te, REGION_te, DB_te, "TEST pool (Male2+Male3, held out) — does it generalize?")
print(f"The border effect is real and reproducible across genuinely different brains "
      f"(train: +{100*prize_tr:.2f}%, test: +{100*prize_te:.2f}%) — if anything slightly larger "
      f"in the held-out brains, not an artifact of the pooled training data.")

## Making the border effect clear on the TEST pool alone

The table above already shows a same/cross gap in the held-out brains, but as a single point estimate
per bin. Bootstrap over random subsets of the **TEST pool only** (Male2+Male3, never touched by any
training statistic) to put a confidence band on it: how much does the same-region vs. cross-region gap
actually move around under resampling, and does it reliably exclude zero at every distance?

In [ ]:
N_BOOT_BORDER = 16
rng_border = np.random.default_rng(7)
same_draws = {b: [] for b in range(len(EDG) - 1)}
cross_draws = {b: [] for b in range(len(EDG) - 1)}
for draw in range(N_BOOT_BORDER):
    idx = rng_border.choice(NTE, 2500, replace=False)
    mm, z, reg = anch_mm_te[idx], Z_te[idx], REGION_te[idx]
    I, J = np.triu_indices(len(idx), 1)
    dd = np.linalg.norm(mm[I] - mm[J], axis=1)
    sel = dd <= 2.0
    I, J, dd = I[sel], J[sel], dd[sel]
    S = (z[I] * z[J]).mean(1)
    XR = reg[I] != reg[J]
    for b in range(len(EDG) - 1):
        m = (dd >= EDG[b]) & (dd < EDG[b+1])
        same_draws[b].append(S[m & ~XR].mean())
        cross_draws[b].append(S[m & XR].mean())

ctr = 0.5 * (EDG[1:] + EDG[:-1])
same_mean = np.array([np.mean(same_draws[b]) for b in range(len(EDG)-1)])
same_lo = np.array([np.percentile(same_draws[b], 2.5) for b in range(len(EDG)-1)])
same_hi = np.array([np.percentile(same_draws[b], 97.5) for b in range(len(EDG)-1)])
cross_mean = np.array([np.mean(cross_draws[b]) for b in range(len(EDG)-1)])
cross_lo = np.array([np.percentile(cross_draws[b], 2.5) for b in range(len(EDG)-1)])
cross_hi = np.array([np.percentile(cross_draws[b], 97.5) for b in range(len(EDG)-1)])

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(ctr, same_mean, "o-", color="#1baf7a", lw=2, label="same-region")
ax.fill_between(ctr, same_lo, same_hi, color="#1baf7a", alpha=.2)
ax.plot(ctr, cross_mean, "o-", color="#e34948", lw=2, label="cross-region")
ax.fill_between(ctr, cross_lo, cross_hi, color="#e34948", alpha=.2)
ax.set_xlabel("distance (mm)"); ax.set_ylabel("mean correlation")
ax.set_title(f"Border effect on the TEST pool alone (Male2+Male3, held out)\n"
             f"{N_BOOT_BORDER} bootstrap draws of 2500/{NTE} anchors, 95% CI")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

print(f"{'d (mm)':<11}{'C_same [95% CI]':>26}{'C_cross [95% CI]':>26}{'gap [95% CI]':>24}{'excl. 0':>9}")
for b in range(len(EDG) - 1):
    s, c = np.array(same_draws[b]), np.array(cross_draws[b])
    g = s - c
    s_ci, c_ci, g_ci = np.percentile(s, [2.5, 97.5]), np.percentile(c, [2.5, 97.5]), np.percentile(g, [2.5, 97.5])
    col_s = f"{s.mean():.3f} [{s_ci[0]:.3f},{s_ci[1]:.3f}]"
    col_c = f"{c.mean():.3f} [{c_ci[0]:.3f},{c_ci[1]:.3f}]"
    col_g = f"{g.mean():.3f} [{g_ci[0]:.3f},{g_ci[1]:.3f}]"
    print(f"{f'{EDG[b]}-{EDG[b+1]}':<11}{col_s:>26}{col_c:>26}{col_g:>24}{'yes' if g_ci[0] > 0 else 'no':>9}")

## Locking a kernel by lengthscale sweep — on genuine held-out correlation

Same idea as before, but now the "held-out" split is by **brain**, not by random fold: fit an exact GP
(closed-form Cholesky solve) on the 3,000 TRAIN anchors, predict at the 3,000 TEST anchors, score
Pearson correlation against their true (held-out-brain) values, median over 178 lipids. No k-fold
needed — the brain split *is* the held-out set, once.

**Outputscale still needs no separate sweep** (shown earlier): the GP posterior mean only depends on
$\lambda=\sigma_n^2/\sigma_f^2$, so sweeping $\lambda$ already covers it. $\nu=1.5$ for Euclidean and
$\nu=2$ for the manifold kernel, fixed from an earlier per-lipid check; only $\ell$ is grid-searched.

In [ ]:
from scipy.special import kv, gamma as gamma_fn

def rho_euclid_mat(nu, ls, Dmat):
    if nu == 0.5:
        return np.exp(-Dmat / ls)
    z = np.sqrt(2 * nu) * np.maximum(Dmat, 1e-12) / ls
    return (2 ** (1 - nu) / gamma_fn(nu)) * z ** nu * kv(nu, z)

NU_EUC, NU_MAN = 1.5, 2.0

# ---- lengthscale lock: picks each kernel's best held-out-correlation lengthscale by
# sweeping a euclidean and manifold grid x a lambda (ridge) grid, across all 3 graphs
# (K=1300 manifold matmuls x 11 lengthscales x 3 graphs) -- computed once and cached;
# loaded here directly.
LOCK_CACHE = SWEEP_CACHE / f"fold{FOLD}_lock_cache.npz"
if not LOCK_CACHE.exists():
    raise FileNotFoundError(f"{LOCK_CACHE} missing -- NO producer script exists in the repo -- this one is a sweep result whose code is gone; copy it from manifold/analysis/")
_c = np.load(LOCK_CACHE, allow_pickle=True)
best_euc = (float(_c["euc_ell"]), float(_c["euc_med"]), float(_c["euc_lam"]), _c["euc_corr"])
best_man = {g: (float(_c[f"man_{g}_ell"]), float(_c[f"man_{g}_med"]),
                float(_c[f"man_{g}_lam"]), _c[f"man_{g}_corr"]) for g in GRAPHS}
LOCKED_EUC = (NU_EUC, best_euc[0])
LOCKED_MAN = (NU_MAN, best_man["atlas x50"][0])

print(f"LOCKED euclidean: nu={NU_EUC}, ell={best_euc[0]}, held-out corr = {best_euc[1]:.4f}")
for g in GRAPHS:
    print(f"LOCKED {g:<24} nu={NU_MAN}, ell={best_man[g][0]:<5} held-out corr = {best_man[g][1]:.4f}")

**Euclidean still wins, decisively, under genuine cross-subject generalization.** Its held-out
optimum beats every manifold configuration's own optimum. `atlas x50` remains the best of the three
graphs; pruning hurts, `faiss` (no atlas information at all) is worst. The margins are smaller than
the old pooled-random-CV numbers suggested (as with "which distance explains" above), but the
direction is exactly the same, now on the split that actually matters.

## The same comparison, on a shared physical mm axis

The earlier correlogram ("which distance explains the lipid data", panel (a)) plotted each metric
against its own native distance — mm for Euclidean, an unrelated diffusion-distance scale for the
manifold graph — so the two curves were never actually comparable, and the manifold one collapsed
into a sliver near zero. Here both LOCKED models' own implied correlation function $\rho(d)$ is
evaluated at the *same real physical mm distances* (TEST pool pairs, ≤2mm), so the shapes can be read
directly against each other.

Both model curves sit above the empirical one everywhere — expected, not an error: $\rho(d)$ is the
prior signal correlation, while the empirical curve is a noise-attenuated raw single-sample estimate.
Compare the two colored model curves to each other, not to black.

In [ ]:
idx0 = np.random.default_rng(0).choice(NTE, 2500, replace=False)
I0, J0 = np.triu_indices(len(idx0), 1)
mm0 = anch_mm_te[idx0]
dd0 = np.linalg.norm(mm0[I0] - mm0[J0], axis=1)
keep0 = dd0 <= 2.0
I0, J0, dd0 = I0[keep0], J0[keep0], dd0[keep0]
S0 = np.nanmean(Z_te[idx0][I0] * Z_te[idx0][J0], axis=1)

vec0 = VEC_TE["atlas x50"][idx0]
Sd = (2 * LOCKED_MAN[0] / LOCKED_MAN[1] ** 2 + EIGVAL["atlas x50"]) ** (-LOCKED_MAN[0]); Sd /= Sd.sum()
diag0 = (vec0 * Sd * vec0).sum(1)
K_man_pairs = ((vec0[I0] * Sd) * vec0[J0]).sum(1) / np.sqrt(diag0[I0] * diag0[J0])

edges_mm = np.linspace(0, 2.0, 13)
ctrs, emps, eucs, mans = [], [], [], []
for k in range(len(edges_mm) - 1):
    m = (dd0 >= edges_mm[k]) & (dd0 < edges_mm[k + 1])
    ctrs.append(dd0[m].mean()); emps.append(S0[m].mean())
    eucs.append(rho_euclid_mat(*LOCKED_EUC, dd0[m].mean()))
    mans.append(K_man_pairs[m].mean())

fig, ax = plt.subplots(figsize=(6.5, 4.6))
ax.plot(ctrs, emps, "o-", color="#111", lw=2, label="empirical (TEST pool)")
ax.plot(ctrs, eucs, "o-", color=COL["euclidean"], lw=2, label=f"euclidean model (ell={LOCKED_EUC[1]})")
ax.plot(ctrs, mans, "o-", color=COL["laplacian · atlas x50"], lw=2,
        label=f"manifold model, atlas x50 (ell={LOCKED_MAN[1]})")
ax.axhline(0, color="#999", lw=.8, ls=":")
ax.set_xlabel("distance (mm)"); ax.set_ylabel("correlation")
ax.set_title("Correlogram on a shared physical mm axis — LOCKED models, TEST pool")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

print(f"model-implied correlation at d=2mm: euclidean={rho_euclid_mat(*LOCKED_EUC, 2.0):.4f}   "
      f"manifold·atlas x50={mans[-1]:.4f}")

## Does either kernel's own model actually capture the border effect?

The border-prize section above showed a real gap in the *data*: same-region pairs correlate more than
cross-region pairs, at every physical distance. But does either kernel's own fitted function even have
the *capacity* to represent that gap? Euclidean's $\rho(d)$ is a pure function of physical distance —
structurally, it cannot distinguish a same-region pair from a cross-region pair at the same distance,
so whatever "gap" its own model shows is not a real effect, just noise from slightly different average
distances between the two subsets. The manifold kernel's $\rho$ is built from the atlas-weighted graph,
so it genuinely *can* differ for same- vs. cross-region pairs at equal physical distance. Using the
LOCKED models above — fit purely to maximize held-out correlation, never told about this specific
pattern — how much of the true border gap does each one's own implied correlation function actually
reproduce?

In [ ]:
I, J = np.triu_indices(NTR, 1)
dd = np.linalg.norm(anch_mm_tr[I] - anch_mm_tr[J], axis=1)
sel = dd <= 2.0
I, J, dd = I[sel], J[sel], dd[sel]
XR = REGION_tr[I] != REGION_tr[J]
S_emp = (Z_tr[I] * Z_tr[J]).mean(1)

rho_euc_model = rho_euclid_mat(*LOCKED_EUC, dd)

vec = VEC_TR["atlas x50"]
Sd = (2 * LOCKED_MAN[0] / LOCKED_MAN[1] ** 2 + EIGVAL["atlas x50"]) ** (-LOCKED_MAN[0]); Sd /= Sd.sum()
diag = (vec * Sd * vec).sum(1)
rho_man_model = ((vec[I] * Sd) * vec[J]).sum(1) / np.sqrt(diag[I] * diag[J])

print(f"{'d (mm)':<11}{'gap: empirical':>16}{'gap: euclid model':>19}{'gap: manifold model':>21}{'% of true gap':>15}")
gaps_emp, gaps_man = [], []
for b in range(len(EDG) - 1):
    m = (dd >= EDG[b]) & (dd < EDG[b+1])
    g_emp = S_emp[m & ~XR].mean() - S_emp[m & XR].mean()
    g_euc = rho_euc_model[m & ~XR].mean() - rho_euc_model[m & XR].mean()
    g_man = rho_man_model[m & ~XR].mean() - rho_man_model[m & XR].mean()
    gaps_emp.append(g_emp); gaps_man.append(g_man)
    print(f"{f'{EDG[b]}-{EDG[b+1]}':<11}{g_emp:>16.3f}{g_euc:>19.5f}{g_man:>21.3f}{100*g_man/g_emp:>14.0f}%")
print(f"\nmean gap captured by the manifold model: {100*np.mean(gaps_man)/np.mean(gaps_emp):.0f}%   "
      f"(euclidean's own model gap is ~0 in every bin, by construction -- its kernel cannot represent "
      f"a same/cross-region difference at any distance, no matter how it's fit)")

ctr_edg = 0.5 * (EDG[1:] + EDG[:-1])
same_emp = [S_emp[(dd>=EDG[i])&(dd<EDG[i+1])&~XR].mean() for i in range(len(EDG)-1)]
cross_emp = [S_emp[(dd>=EDG[i])&(dd<EDG[i+1])&XR].mean() for i in range(len(EDG)-1)]
same_man = [rho_man_model[(dd>=EDG[i])&(dd<EDG[i+1])&~XR].mean() for i in range(len(EDG)-1)]
cross_man = [rho_man_model[(dd>=EDG[i])&(dd<EDG[i+1])&XR].mean() for i in range(len(EDG)-1)]
euc_line = [rho_euc_model[(dd>=EDG[i])&(dd<EDG[i+1])].mean() for i in range(len(EDG)-1)]

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(ctr_edg, same_emp, "o-", color="#111", lw=2, label="empirical, same-region")
ax.plot(ctr_edg, cross_emp, "o--", color="#111", lw=2, alpha=.5, label="empirical, cross-region")
ax.plot(ctr_edg, same_man, "o-", color=COL["laplacian · atlas x50"], lw=2, label="manifold model, same-region")
ax.plot(ctr_edg, cross_man, "o--", color=COL["laplacian · atlas x50"], lw=2, alpha=.6, label="manifold model, cross-region")
ax.plot(ctr_edg, euc_line, "s:", color=COL["euclidean"], lw=2, label="euclidean model (identical same/cross, by construction)")
ax.set_xlabel("distance (mm)"); ax.set_ylabel("correlation")
ax.set_title("Only the manifold kernel's own model can represent the border drop")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

**The manifold kernel captures roughly 61% of the true border gap on its own, at every distance
band; Euclidean's is ~0% everywhere, by construction.** This is a structural difference, not a
fitting difference — no lengthscale, however chosen, lets Euclidean's $\rho(d)$ depend on anything
but $d$. The manifold kernel doesn't reproduce the border gap *exactly* (61%, not 100%), but it's the
only one of the two whose functional form is even capable of representing it at all.

**A more literal reading — "distance from the border" itself — is murkier.** Restricting to
cross-region pairs only and binning by how close the *nearer* point sits to any region boundary
(rather than same/cross composition), both kernels' implied correlation just tracks the same mild
decline with physical distance; neither one's own model responds distinctively to *how close* a
crossing pair sits to the boundary line itself, only to *whether* it crosses at all. That table is
below for completeness, but the same/cross-by-distance comparison above is the cleaner result.

In [ ]:
# appendix: literal "distance from the border" -- cross-region pairs only, binned by how close
# the NEARER of the two points sits to any region boundary (DB = distance-to-nearest-boundary, mm).
border_prox = np.minimum(DB_tr[I][XR], DB_tr[J][XR])
dd_xr, S_emp_xr = dd[XR], S_emp[XR]
rho_euc_xr, rho_man_xr = rho_euc_model[XR], rho_man_model[XR]

PROX_EDG = [0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 0.9, 1.3, 2.55]
print(f"{'border dist (mm)':<18}{'n':>8}{'mean pair dd':>13}{'empirical':>11}{'euclid model':>14}{'manifold model':>16}")
for k in range(len(PROX_EDG) - 1):
    m = (border_prox >= PROX_EDG[k]) & (border_prox < PROX_EDG[k+1])
    if m.sum() == 0:
        continue
    print(f"{PROX_EDG[k]}-{PROX_EDG[k+1]:<12}{m.sum():>8,}{dd_xr[m].mean():>13.3f}"
          f"{S_emp_xr[m].mean():>11.4f}{rho_euc_xr[m].mean():>14.4f}{rho_man_xr[m].mean():>16.4f}")

## Does the border prior actually help *predictions*, not just priors?

The "61% of the border gap" result above is about the kernels' own **prior** covariance — it shows the
manifold *can* represent a same/cross drop and Euclidean structurally cannot. But a prior shift only
matters if it survives conditioning on training data and improves held-out predictions. Two predictive
tests, both fitting the LOCKED kernels on the full 3,000-anchor TRAIN pool (best-$\lambda$ from the
lock cache) and predicting the held-out brains:

- **(2b) Cross-border contrasts** — the sharpest test of the boundary prior. For each held-out
  cross-region pair $(i,j)$ within 2mm, form the observed across-boundary jump $\Delta y=y_i-y_j$ and
  the predicted jump $\Delta\hat y=\hat y_i-\hat y_j$, and score $\operatorname{corr}(\Delta\hat
  y,\Delta y)$ and $\operatorname{MSE}(\Delta\hat y-\Delta y)$. Binned by pair proximity to the
  boundary, $\min(d_i,d_j)$ where $d$ is distance-to-nearest-boundary. This isolates the one thing the
  boundary prior should get right.

- **(1) Whole-point prediction** — what the deployed model is actually scored on. Predict each held-out
  point and stratify its MSE and correlation by the point's own distance to a boundary.

**Significance for (2b) is taken over the 178 lipids, never over pairs** — pairs are massively
dependent (each point sits in thousands), so pair counts are descriptive only.

In [ ]:
# ---- fit each LOCKED kernel on the full 3000-anchor TRAIN pool, predict all TEST anchors, once.
LAM_E, LAM_M = best_euc[2], best_man["atlas x50"][2]
Ktt_e = rho_euclid_mat(*LOCKED_EUC, cdist(anch_mm_tr, anch_mm_tr))
Kst_e = rho_euclid_mat(*LOCKED_EUC, cdist(anch_mm_te, anch_mm_tr))
_vt, _ve = VEC_TR["atlas x50"], VEC_TE["atlas x50"]
_Sd = (2 * LOCKED_MAN[0] / LOCKED_MAN[1] ** 2 + EIGVAL["atlas x50"]) ** (-LOCKED_MAN[0]); _Sd /= _Sd.sum()
_dtr, _dte = (_vt * _Sd * _vt).sum(1), (_ve * _Sd * _ve).sum(1)
Ktt_m = ((_vt * _Sd) @ _vt.T) / np.sqrt(np.outer(_dtr, _dtr))
Kst_m = ((_ve * _Sd) @ _vt.T) / np.sqrt(np.outer(_dte, _dtr))
PRED_E = Kst_e @ np.linalg.solve(Ktt_e + LAM_E * np.eye(NTR), Z_tr)
PRED_M = Kst_m @ np.linalg.solve(Ktt_m + LAM_M * np.eye(NTR), Z_tr)

# ---- (2b) predicted cross-border CONTRAST, binned by pair proximity to boundary min(d_i,d_j).
Ic, Jc = np.triu_indices(NTE, 1)
ddc = np.linalg.norm(anch_mm_te[Ic] - anch_mm_te[Jc], axis=1)
xb = (ddc <= 2.0) & (REGION_te[Ic] != REGION_te[Jc])
Ic, Jc = Ic[xb], Jc[xb]
prox = np.minimum(DB_te[Ic], DB_te[Jc])
dY = Z_te[Ic] - Z_te[Jc]
dPE, dPM = PRED_E[Ic] - PRED_E[Jc], PRED_M[Ic] - PRED_M[Jc]

PROX_BINS = [0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 2.0]
cb_lab, cb_dc, cb_dm, cb_pc, cb_pm = [], [], [], [], []
print(f"{'proximity':<12}{'npairs':>8}{'Δcorr M-E':>11}{'corr wins':>11}{'p(>0)':>9}"
      f"{'ΔMSE M-E':>11}{'MSE wins':>11}{'p(<0)':>9}")
for lo, hi in zip(PROX_BINS[:-1], PROX_BINS[1:]):
    b = (prox >= lo) & (prox < hi)
    if b.sum() < 10:
        continue
    dcorr = np.array([np.corrcoef(dPM[b, l], dY[b, l])[0, 1] - np.corrcoef(dPE[b, l], dY[b, l])[0, 1]
                      for l in range(L)])
    dmse = np.array([np.mean((dPM[b, l] - dY[b, l]) ** 2) - np.mean((dPE[b, l] - dY[b, l]) ** 2)
                     for l in range(L)])
    pc = wilcoxon(dcorr, alternative="greater").pvalue
    pm = wilcoxon(dmse, alternative="less").pvalue
    cb_lab.append(f"{lo:g}-{hi:g}"); cb_dc.append(np.median(dcorr)); cb_dm.append(np.median(dmse))
    cb_pc.append(pc); cb_pm.append(pm)
    print(f"{f'{lo:g}-{hi:g}mm':<12}{int(b.sum()):>8}{np.median(dcorr):>+11.4f}"
          f"{f'{(dcorr>0).sum()}/{L}':>11}{pc:>9.1e}{np.median(dmse):>+11.4f}"
          f"{f'{(dmse<0).sum()}/{L}':>11}{pm:>9.1e}")

xx = np.arange(len(cb_lab))
C_MAN, C_EUC = COL["laplacian · atlas x50"], COL["euclidean"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
ax = axes[0]
ax.bar(xx, cb_dc, .62, color=[C_MAN if v > 0 else C_EUC for v in cb_dc], edgecolor="white")
for x, v, p in zip(xx, cb_dc, cb_pc):
    if p < 0.05: ax.text(x, v + (0.0006 if v > 0 else -0.0006), "*", ha="center",
                         va="bottom" if v > 0 else "top", fontsize=13)
ax.axhline(0, color=INK2, lw=.8); ax.set_xticks(xx); ax.set_xticklabels(cb_lab, fontsize=8)
ax.set_xlabel("pair proximity to boundary  min(d_i, d_j)  (mm)")
ax.set_ylabel("Δ contrast correlation (manifold − euclidean)")
ax.set_title("(a) Across-boundary jump — pattern (↑ = manifold better)")
ax = axes[1]
ax.bar(xx, cb_dm, .62, color=[C_MAN if v < 0 else C_EUC for v in cb_dm], edgecolor="white")
for x, v, p in zip(xx, cb_dm, cb_pm):
    if p < 0.05: ax.text(x, v + (-0.002 if v < 0 else 0.002), "*", ha="center",
                         va="top" if v < 0 else "bottom", fontsize=13)
ax.axhline(0, color=INK2, lw=.8); ax.set_xticks(xx); ax.set_xticklabels(cb_lab, fontsize=8)
ax.set_xlabel("pair proximity to boundary  min(d_i, d_j)  (mm)")
ax.set_ylabel("Δ contrast MSE (manifold − euclidean)")
ax.set_title("(b) Across-boundary jump — magnitude (↓ = manifold better)")
fig.suptitle("(2b) The boundary prior helps predict cross-border changes — but only near the boundary "
             f"(* = p<0.05 over {L} lipids)", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

**The boundary prior is real and it does help predictions — but only for pairs that straddle the
boundary tightly.** Within ~0.2mm of the boundary the manifold predicts the across-boundary jump
better than Euclidean on **both** metrics: contrast correlation improves by ~0.013 at 0.15–0.20mm
(≈152/178 lipids, p≈10⁻²¹) and contrast MSE drops (≈159/178, p≈10⁻²⁷). A weaker advantage persists to
0.2–0.3mm. **Beyond ~0.3–0.4mm the advantage reverses** — the two points no longer straddle a boundary
tightly, there is nothing for the atlas prior to correct, and the manifold's broader spectral
constraints start to cost it. This is the concrete, conditioned, held-out evidence that the manifold
*uses* its anatomical prior where it should — the (2a) capacity result above is not just a property of
the prior, it survives training and improves cross-border predictions.

In [ ]:
# ---- (1) whole-point held-out prediction (reuses PRED_E / PRED_M from above), stratified by each
# TEST point's OWN distance to a boundary. This is what a deployed model is actually scored on.
def _med_corr(P, mask):
    return np.nanmedian([np.corrcoef(P[mask, l], Z_te[mask, l])[0, 1] for l in range(L)])
mse_e_pt = np.nanmean((PRED_E - Z_te) ** 2, axis=1)
mse_m_pt = np.nanmean((PRED_M - Z_te) ** 2, axis=1)

PT_BINS = [0.1, 0.2, 0.3, 0.5, 0.8, np.inf]
pt_lab, pt_ce, pt_cm, pt_me, pt_mm = [], [], [], [], []
print(f"{'border dist':<13}{'n':>7}{'corr E':>9}{'corr M':>9}{'M-E':>9}{'MSE E':>9}{'MSE M':>9}{'M-E':>9}")
for lo, hi in zip(PT_BINS[:-1], PT_BINS[1:]):
    b = (DB_te >= lo) & (DB_te < hi)
    if b.sum() < 10:
        continue
    ce, cm = _med_corr(PRED_E, b), _med_corr(PRED_M, b)
    me, mm = mse_e_pt[b].mean(), mse_m_pt[b].mean()
    lab = f"{lo:g}-{hi:g}" if np.isfinite(hi) else f">{lo:g}"
    pt_lab.append(lab); pt_ce.append(ce); pt_cm.append(cm); pt_me.append(me); pt_mm.append(mm)
    print(f"{lab:<13}{int(b.sum()):>7}{ce:>9.4f}{cm:>9.4f}{cm-ce:>+9.4f}{me:>9.4f}{mm:>9.4f}{mm-me:>+9.4f}")

xx = np.arange(len(pt_lab))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
ax = axes[0]
ax.plot(xx, pt_ce, "o-", color=COL["euclidean"], lw=2, label="euclidean")
ax.plot(xx, pt_cm, "o-", color=COL["laplacian · atlas x50"], lw=2, label="manifold")
ax.set_xticks(xx); ax.set_xticklabels(pt_lab, fontsize=8)
ax.set_xlabel("test point's distance to boundary (mm)")
ax.set_ylabel("held-out correlation (median over lipids)")
ax.set_title("(a) Whole-point prediction — correlation"); ax.legend(fontsize=8)
ax = axes[1]
ax.plot(xx, pt_me, "o-", color=COL["euclidean"], lw=2, label="euclidean")
ax.plot(xx, pt_mm, "o-", color=COL["laplacian · atlas x50"], lw=2, label="manifold")
ax.set_xticks(xx); ax.set_xticklabels(pt_lab, fontsize=8)
ax.set_xlabel("test point's distance to boundary (mm)")
ax.set_ylabel("held-out MSE (lower = better)")
ax.set_title("(b) Whole-point prediction — MSE"); ax.legend(fontsize=8)
fig.suptitle("(1) But whole-point prediction: the manifold loses in every stratum "
             "(least near the border, worst in the interior)", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

**...yet for whole-point prediction the manifold loses in every stratum — it is just *least* bad near
the border.** The correlation deficit is smallest right at the boundary (~−0.003) and grows steadily
into the interior (to ~−0.02 for the deepest points); MSE tells the same story. The localized
cross-border benefit from (2b) is genuine, but a *point's* prediction is dominated by same-side
interpolation from its many nearby same-region neighbors, not by the handful of across-boundary pairs —
so the boundary correction moves the whole-point number very little, while the manifold's broad
interior over-smoothing handicap (quantified in the next sections) applies everywhere and dominates.

**The synthesis, in one line:** the manifold's anatomical prior is real, conditioned-in, and helps
*exactly* where it should (tight cross-boundary contrasts) — but that benefit is spatially confined and
is outweighed by a distance-metric handicap that costs it across the entire interior. A real, localized
boundary gain is fully compatible with losing the overall predictive comparison, which is what happens
here.

## Being honest about Nystrom: the same sweep with the real kernel code

Every number above is a hand-rolled numpy reimplementation, and — to keep that code simple — every
TRAIN/TEST anchor is snapped to the nearest exact graph node. That means the manifold kernel's Nystrom
out-of-sample extension is never actually exercised: on-graph points always take the exact
eigenvector-lookup path. Real deployment evaluates at genuine, un-snapped voxel/cell positions, which
almost never land exactly on a graph node — so it leans on Nystrom for nearly every point.

`manifold/analysis/sweep_real_kernels.py` closes that gap: it re-pools TRAIN/TEST using the **true continuous
cell centroid** (never snapped), and scores them with the **actual deployed kernel classes** —
`gpytorch.kernels.MaternKernel` and `manifold_gp.kernels.RiemannMaternKernel` — not a reimplementation.
The manifold kernel's own `.features()` does the real Nystrom interpolation and bump-function gating
internally. Runs on GPU; caches its (small) results here. (Verified separately: the real kernel and the
hand-rolled formula above agree to ~1e-6 for on-graph points, confirming the hand-rolled math itself was
never the issue — only the never-off-graph evaluation was.)

In [ ]:
REAL_CACHE = SWEEP_CACHE / f"fold{FOLD}_real_kernel_sweep.npz"
if not REAL_CACHE.exists():
    raise FileNotFoundError(
        f"{REAL_CACHE.name} missing. Build it once (GPU-accelerated, real kernel classes,\n -- rebuild with:  python manifold/analysis/sweep_real_kernels.py"
        f"real Nystrom interpolation on un-snapped centroids, all {len(GRAPHS)} manifold graphs):\n\n"
        f"    python manifold/analysis/sweep_real_kernels.py\n")

real = np.load(REAL_CACHE, allow_pickle=True)
euc_corr_real = real["euc_corr"]
euc_ell_real, euc_med_real = float(real["euc_ell"]), np.nanmedian(euc_corr_real)
print(f"(built on {real['device']})\n")

print(f"{'kernel':<24}{'hand-rolled (snapped-to-node)':>32}{'real kernel (real Nystrom)':>30}")
col_hand = f"ell={LOCKED_EUC[1]}, corr={best_euc[1]:.4f}"
col_real = f"ell={euc_ell_real}, corr={euc_med_real:.4f}"
print(f"{'euclidean':<24}{col_hand:>32}{col_real:>30}")

man_corr_real, man_med_real = {}, {}
for g in GRAPHS:
    key = g.replace(" ", "_")
    man_corr_real[g] = real[f"man_{key}_corr"]
    ell_real = float(real[f"man_{key}_ell"])
    man_med_real[g] = np.nanmedian(man_corr_real[g])
    col_hand = f"ell={best_man[g][0]}, corr={best_man[g][1]:.4f}"
    col_real = f"ell={ell_real}, corr={man_med_real[g]:.4f}"
    print(f"{'manifold · ' + g:<24}{col_hand:>32}{col_real:>30}")

best_graph_real = max(man_med_real, key=man_med_real.get)
print(f"\nbest manifold graph (real kernel): {best_graph_real}  (median corr = {man_med_real[best_graph_real]:.4f})")
print(f"euclidean wins by {euc_med_real - man_med_real[best_graph_real]:.4f} correlation under the real, "
      f"Nystrom-honest treatment (was {best_euc[1] - best_man['atlas x50'][1]:.4f} hand-rolled)")

wins = euc_corr_real > man_corr_real[best_graph_real]
print(f"euclidean wins on {int(np.nansum(wins))}/{len(real['lipids'])} lipids individually "
      f"(vs its best real-kernel manifold competitor, {best_graph_real})")

**The real, Nystrom-honest kernels agree with the hand-rolled ones on the headline result.** All
held-out correlations move up slightly (true continuous cell centroids are a marginally better
representation of the data than snapping to the nearest graph node, benefiting every kernel roughly
equally) — but Euclidean's win over the *best* manifold graph is essentially unchanged. This was never
an artifact of the notebook's simplified on-graph-only evaluation: swapping in the actual deployed
kernel classes, with genuine Nystrom interpolation exercised on every point, reproduces the same result.

**One real nuance did change, though: pruning stops looking harmful.** Hand-rolled/snapped, `atlas x50
+ prune .95` clearly trailed the unpruned graph (0.7367 vs 0.7449) — consistent with the
disconnected-component story elsewhere in this notebook. Under the real kernel with genuine Nystrom
interpolation, the two are essentially tied (0.7479 vs 0.7481). A plausible reason: Nystrom
interpolation blends each off-graph query across its several nearest graph nodes rather than reading a
single exact eigenvector row, which can smooth over some of the damage a few disconnected components do
to the raw on-graph spectrum. `faiss` (no atlas information) remains clearly the worst of the three
manifold graphs either way.

## Is "decays too fast" the real reason — or is there a hidden cost to fixing it?

Sweep $\ell$ for the manifold kernel once more, tracking **both** the tail-retention-at-2mm (matched to
the TRAIN pool's own decay rate) **and** fold-2 held-out correlation side by side.

In [ ]:
I5, J5 = np.triu_indices(NTR, 1)
dd5 = np.linalg.norm(anch_mm_tr[I5] - anch_mm_tr[J5], axis=1)
sel5 = dd5 <= 2.0
I5, J5, dd5 = I5[sel5], J5[sel5], dd5[sel5]
XR5 = REGION_tr[I5] != REGION_tr[J5]
S_emp_tr = (Z_tr[I5] * Z_tr[J5]).mean(1)

def retain_2mm(v):
    row = np.array([v[(dd5 >= EDG[i]) & (dd5 < EDG[i+1]) & ~XR5].mean() for i in range(len(EDG) - 1)])
    return row[-1] / row[0]

emp_retain = retain_2mm(S_emp_tr)
print(f"EMPIRICAL retain@2mm (TRAIN pool, target) = {emp_retain:.3f}\n")

# ---- decay-rate-vs-held-out-cost sweep: reruns the K=1300 manifold solve at ell in
# {1,2,3,4,5,6,6.5,7,8,10,12,15,20} (nu=NU_MAN, atlas x50 graph) -- computed once and
# cached; loaded here directly.
DECAY_CACHE = SWEEP_CACHE / f"fold{FOLD}_decay_cost_cache.npz"
if not DECAY_CACHE.exists():
    raise FileNotFoundError(f"{DECAY_CACHE} missing -- NO producer script exists in the repo -- this one is a sweep result whose code is gone; copy it from manifold/analysis/")
_c = np.load(DECAY_CACHE, allow_pickle=True)
sweep = list(zip(_c["ls"].tolist(), _c["r2mm"].tolist(), _c["ho"].tolist()))

print(f"{'ell':>6}{'retain@2mm':>12}{'|gap to target|':>17}{'held-out corr':>16}\n")
for ls, r2mm, ho in sweep:
    print(f"{ls:>6}{r2mm:>12.3f}{abs(r2mm - emp_retain):>17.3f}{ho:>16.4f}")

best_match = min(sweep, key=lambda r: abs(r[1] - emp_retain))
best_ho = max(sweep, key=lambda r: r[2])
print(f"\nbest DECAY-RATE match : ell={best_match[0]} (retain={best_match[1]:.3f}) -> held-out corr = {best_match[2]:.4f}")
print(f"best HELD-OUT ell     : ell={best_ho[0]} (retain={best_ho[1]:.3f}) -> held-out corr = {best_ho[2]:.4f}")
print(f"COST of forcing the correct decay rate: {best_ho[2] - best_match[2]:.4f} held-out correlation")

**Confirmed again, under real cross-subject generalization.** The $\ell$ that matches the TRAIN
pool's own decay rate is within noise of the $\ell$ that maximizes fold-2 held-out correlation — the
cost of insisting on the correct decay rate is a few ten-thousandths, not the real gap to Euclidean.
"Decays too fast" (at hand-picked lengthscales) was never the bottleneck, on either evaluation design.

## Where does the manifold kernel's weight actually live?

The kernel is $K(i,j)=\sum_k S(\lambda_k)\,\varphi_k(i)\varphi_k(j)$ — a smooth spectral filter over the
graph's own eigenvalues. Two ways to ask "how many of the (now 1,300) modes actually matter": modes
needed to reach 90% of the total weight (a compression question), and the **participation ratio**
$\big(\sum_k S(\lambda_k)\big)^2/\sum_k S(\lambda_k)^2$ — the effective number of independent,
comparably-weighted directions, which is what actually caps a GP's prediction complexity.

In [ ]:
lam_atlas = EIGVAL["atlas x50"]

def spectral_weight(nu, ls):
    Sd = (2 * nu / ls ** 2 + lam_atlas) ** (-nu)
    return Sd / Sd.sum()

def eff_modes(w):
    return 1.0 / (w ** 2).sum()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
CONFIGS = [(NU_MAN, 5.0, "#1baf7a", "atlas x50, ν=2, ℓ=5"),
           (NU_MAN, LOCKED_MAN[1], "#0d366b", f"atlas x50, ν=2, ℓ={LOCKED_MAN[1]} (LOCKED)")]

ax = axes[0]
ax.plot(np.arange(1, len(lam_atlas)+1), lam_atlas, color="#333", lw=1.8)
ax.set_xlabel("mode index k"); ax.set_ylabel("raw eigenvalue λ_k")
ax.set_title(f"(a) Raw graph spectrum (atlas x50, K={len(lam_atlas)})\nnot kernel-dependent")

ax = axes[1]
for nu, ls, c, lbl in CONFIGS:
    ax.plot(np.arange(1, len(lam_atlas)+1), spectral_weight(nu, ls), color=c, lw=2, label=lbl)
ax.set_yscale("log")
ax.set_xlabel("mode index k"); ax.set_ylabel("normalized weight $S(\\lambda_k)/\\sum S(\\lambda)$")
ax.set_title("(b) Spectral weight per mode (log scale)")
ax.legend(fontsize=8)

ax = axes[2]
for nu, ls, c, lbl in CONFIGS:
    w = np.sort(spectral_weight(nu, ls))[::-1]
    cum = np.cumsum(w)
    eff = eff_modes(w)
    k90 = np.searchsorted(cum, 0.90) + 1
    ax.plot(np.arange(1, len(lam_atlas)+1), cum, color=c, lw=2, label=f"{lbl} (eff. modes={eff:.0f}, 90%@{k90})")
    ax.plot(k90, 0.90, "o", color=c, ms=6)
ax.axhline(0.90, color="#999", lw=.8, ls=":")
ax.set_xlabel("modes kept (ascending λ)"); ax.set_ylabel("cumulative share of total weight")
ax.set_title("(c) Modes needed to capture 90% of the kernel")
ax.legend(fontsize=7.5)

fig.suptitle("The manifold kernel's weight is concentrated in a handful of modes", fontsize=12, y=1.04)
fig.tight_layout()
plt.show()

# --- compare against EUCLIDEAN's empirical Gram-matrix effective rank, fold-2 TRAIN pool ---
K_euc = rho_euclid_mat(*LOCKED_EUC, D["euclidean"]["tr"])
w_euc = np.clip(np.linalg.eigvalsh(K_euc), 0, None)
w_euc_n = w_euc / w_euc.sum()
eff_euc = eff_modes(w_euc_n)

w_man_n = spectral_weight(*LOCKED_MAN)
eff_man = eff_modes(w_man_n)

print(f"MANIFOLD (atlas x50, ν={LOCKED_MAN[0]}, ℓ={LOCKED_MAN[1]}, K={len(lam_atlas)}): effective mode count = {eff_man:.1f}")
print(f"EUCLIDEAN (ν={LOCKED_EUC[0]}, ℓ={LOCKED_EUC[1]}): effective rank of its own {NTR}x{NTR} "
      f"Gram matrix (fold-2 TRAIN pool) = {eff_euc:.1f}")
print(f"\n-> Euclidean has ~{eff_euc/eff_man:.1f}x as many genuinely independent directions to predict with,")
print("   at each kernel's OWN best lengthscale -- a structural gap the decay rate cannot fix.")

**This is the number behind the residual gap.** At its own best lengthscale, the manifold kernel's
posterior mean is built from roughly **~45 effective directions**; Euclidean's is built from roughly
**~80** — about **1.8x richer**. Matching the *average* decay curve only needs the marginal shape to
come out right, and a ~45-mode smooth kernel can do that fine. But predicting a *specific* held-out
point in a *specific new brain* needs enough independent directions to capture real local variation,
and that is where the graph eigenbasis is structurally capped: concentrating weight on enough low modes
to keep the border signal visible leaves few effective directions for anything else. This is not a
lengthscale problem — it is why no lengthscale fixes it, on either evaluation design.

## Why not just shrink $\ell$ to get more effective modes?

If effective rank is the bottleneck, why not use a shorter $\ell$ — more modes get non-negligible
weight, closer to Euclidean's ~80? Check what a **sharp** (untapered) reconstruction from a finite mode
budget actually looks like: the best possible localized "bump" $M$ eigenvectors can build at one
interior point is the **projection kernel** $K_M(i,j)=\sum_{k<M} v_k(i)v_k(j)$ — literally the
sharpest thing $M$ modes can construct, no tapering. Track its width and what it does away from its
peak, across mode counts up to the full $K=1300$.

In [ ]:
vec_gibbs = VEC_TR["atlas x50"]   # (3000, 1300), fold-2 TRAIN pool

rng_g = np.random.default_rng(0)
interior_idx = np.where(DB_tr > 0.4)[0]
picks = rng_g.choice(interior_idx, min(10, len(interior_idx)), replace=False)

MODES_FINE = [10, 30, 50, 100, 200, 300, 500, 700, 1000, 1300]
print(f"{'M':<6}{'1/e-width (mm)':>16}{'worst overshoot':>18}{'frac<0 far-field':>20}")
for M in MODES_FINE:
    Vm = vec_gibbs[:, :M]
    widths, overshoots, fracs_neg = [], [], []
    for i0 in picks:
        d = np.linalg.norm(anch_mm_tr - anch_mm_tr[i0], axis=1)
        k = Vm[i0] @ Vm.T
        peak = k[i0]
        order = np.argsort(d); d_s, k_s = d[order], k[order]
        below = np.where(k_s < peak / np.e)[0]
        w = d_s[below[0]] if len(below) else np.nan
        widths.append(w)
        far = d_s > 2 * w if not np.isnan(w) else d_s > d_s.max()
        if far.sum() > 5:
            overshoots.append((k_s[far] / peak).min())
            fracs_neg.append(100 * (k_s[far] < 0).mean())
    print(f"{M:<6}{np.nanmean(widths):>16.3f}{np.nanmean(overshoots):>18.3f}{np.nanmean(fracs_neg):>19.0f}%")

print(f"\nEuclidean's own locked ell={LOCKED_EUC[1]}: 1/e-width = {LOCKED_EUC[1]:.1f}mm by construction,")
print("and it is NEVER negative, at any lengthscale, ever -- no basis, nothing to ring.")

**Sharpness was never the bottleneck — an untapered mode-budget can build something narrower than
Euclidean's own tuned $\ell$ — but look at what it does past its peak: a large, roughly constant
fraction of far-field pairs come out negative, at every mode count from 10 to the full 1,300.** A real
covariance should decay smoothly toward zero and never flip sign. This is the graph analogue of the
**Gibbs phenomenon**: truncating a basis expansion of a sharply localized target produces ringing beyond
the main lobe that narrows as $M$ grows but does not vanish — a structural property of any finite
eigenbasis. Killing the ringing requires *tapering* the mode weights smoothly rather than cutting off
sharply — exactly what a Matérn $S(\lambda)$ does — but a taper smooth enough to suppress the ringing
is necessarily wide, which is exactly why the manifold kernel needs a long effective $\ell$ instead of
something short like Euclidean's. **Sharp-but-ringing or smooth-but-blurred are the only two options a
finite eigenbasis offers.** Euclidean never faces this choice: distance is a native coordinate, not
something reconstructed from a truncated basis.

---
# Part 2 — Actual trained models

Everything in Part 1 is a closed-form stand-in for a GP (exact kernel-ridge regression on 3,000
anchors) — fast to iterate, but not what actually gets deployed. This part loads the **real, trained**
per-lipid GP checkpoints (`lgp_experiment_per_lipid.py` / `run_lgp_per_lipid.sh`,
variational inducing-point GPs, full training loop) and reads off their genuinely **held-out test
correlation and $R^2$** — `metrics.csv` per run, computed on a separate `test/` split.

Only **5 lipids** have runs for both kernel families at `FOLD-2` (the curated reconstruction subset:
`PC 35:1 PE 38:1`, `PA 36:1`, `LPC 22:6`, `PE O-36:0 PE O-38:3`, `Hex2Cer 40:1;O2`) — a small,
real-deployment sanity check on top of Part 1's broader, closed-form comparison, not a replacement for
it. For the manifold side there are **several different trained configurations** on disk (different
mode counts $K$, fixed vs learned lengthscale, different pruning, and a fully free per-mode spectral
weights variant — the most flexible manifold kernel that exists in this codebase) — every one
currently on disk is included below, so this is the manifold kernel's *best shot*, not a single
cherry-picked run. (Training runs on this machine are ongoing, so the exact count below may grow.)

In [ ]:
import pandas as pd

PER_LIPID_ROOT = Path("/home/casap/mlibra/output/per_lipid")
rows = []
for d in sorted(PER_LIPID_ROOT.iterdir()):
    mf = d / "metrics.csv"
    if not mf.exists():
        continue
    kernel = ("euclidean" if "euclidean" in d.name else
              "spectral"  if "spectral"  in d.name else "manifold")
    df = pd.read_csv(mf)
    df["kernel"], df["run"] = kernel, d.name
    rows.append(df)
long_df = pd.concat(rows, ignore_index=True)

print(f"{long_df['run'].nunique()} runs loaded "
      f"({(long_df['kernel']=='euclidean').sum()//5} euclidean, "
      f"{(long_df['kernel']=='manifold').sum()//5} manifold, "
      f"{(long_df['kernel']=='spectral').sum()//5} spectral), "
      f"{long_df['lipid_name'].nunique()} lipids\n")

print(long_df.pivot_table(index="lipid_name", columns="kernel", values="test_corr",
                           aggfunc=["mean", "max"]).round(3))

print("\nbest-of-euclidean vs best-of-manifold (max test_corr across each family's runs):")
best = long_df.groupby(["lipid_name", "kernel"])["test_corr"].max().unstack()
best["euclidean_wins"] = best["euclidean"] >= best["manifold"]
print(best.round(4))
n_man_final = long_df[long_df.kernel == "manifold"]['run'].nunique()
print(f"\neuclidean wins or ties on {int(best['euclidean_wins'].sum())}/{len(best)} lipids "
      f"(against the BEST of {n_man_final} manifold configurations each)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
lipids = best.index.tolist()
xx = np.arange(len(lipids))
n_euc = long_df[long_df.kernel == "euclidean"]["run"].nunique()
n_man = long_df[long_df.kernel == "manifold"]["run"].nunique()
spectral_best = long_df[long_df.kernel == "spectral"].groupby("lipid_name")["test_corr"].max()

ax.bar(xx - .2, best["euclidean"], .38, color=COL["euclidean"], label=f"best euclidean ({n_euc} runs)",
       edgecolor="white", linewidth=.8)
ax.bar(xx + .2, best["manifold"], .38, color=COL["laplacian · atlas x50"],
       label=f"best manifold ({n_man} runs)", edgecolor="white", linewidth=.8)
ax.plot(xx, spectral_best.loc[lipids],
        "D", color="#e34948", ms=7, label="best learned-spectral-weights (most flexible manifold variant)")
ax.set_xticks(xx); ax.set_xticklabels(lipids, fontsize=8, rotation=15, ha="right")
ax.set_ylabel("held-out test correlation")
ax.set_title(f"Real trained models, FOLD-{FOLD}: Euclidean wins or ties on every lipid,\n"
             f"against the best of {n_man} manifold configurations each")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

**The real, deployed models agree with Part 1's closed-form check.** Euclidean wins or ties on all
5 lipids, against the *best* of every manifold hyperparameter configuration on disk, per lipid —
including a fully learned-spectral-weights variant (free per-mode weighting, the most expressive
version of the manifold kernel that exists in this codebase, per-mode weights instead of a monotone
$S(\lambda)$). The margins are small (consistent with Part 1's ~0.01-0.02 correlation gap), and one
lipid (`Hex2Cer 40:1;O2`) is a genuine tie — but there is no lipid, and no manifold configuration tried,
that beats a plain Euclidean kernel here. This is a small sample (5 of 178 lipids), but it is the
**real, trained, deployed** answer, not a proxy — and it points the same direction as everything in
Part 1.

---
# Part 3 — Three deployed kernels: real hyperparameters, same four checks

The three requested runs, plus one more needed for the last question:

1. `FOLD-2-euclidean-ard-matern-nu1.5-ind1000-5-lr0.05-ep20-lbs1010` — Euclidean, **ARD** (a
   separate lengthscale per axis, per lipid — richer than Part 1's single shared $\ell$).
2. `FOLD-2-learnls-manifold-nu2-K1300-...-prune0.9510` — manifold, learned $\ell$, $K=1300$ modes,
   atlas×50 + prune 0.95.
3. `FOLD-2-spectral-nu2-K1300-...-dn310` — **correction**: this run has `learn_spectral_weights: False`
   in its config. It is the *same* monotone Matérn $S(\lambda)$ as the manifold kernel, just
   parameterized via weight-space variational inference in the eigenbasis (`q_mu`/`q_log_diag_S`), not
   free per-mode weights. The run that actually has one free weight per mode is a sibling,
   `...-learndiff-learnspecw10` (`learn_spectral_weights: True`) — pulled in separately for the
   free-weights question.

**A graph-cache caveat.** These were trained on a remote pod against eigenvectors at `/s3/...`, not
locally reachable. Runs 2's exact graph (`denoise=3, prune=0.95, K=1300`) has a local cache match
(eigendecomposition is a deterministic function of the graph, so this is exact). Runs 3 and
`learnspecw10` (`prune=0.0`) have no exact local match; the closest available
(`atlas x50, rootdissolve, K=1300`, unpruned, no denoise step) is used for their figures below — the
learned $(\nu,\ell)$ values themselves are exact regardless (read directly off the checkpoints), only
the spatial figures for runs 3/free-weights are an approximation.

In [ ]:
import torch, torch.nn.functional as F
import pandas as pd

PER_LIPID_ROOT = Path("/home/casap/mlibra/output/per_lipid")
DEPLOYED_RUNS = {
    "euclidean (ARD)":                 f"FOLD-{FOLD}-euclidean-ard-matern-nu1.5-ind1000-5-lr0.05-ep20-lbs1010",
    "manifold (learned ell)":          f"FOLD-{FOLD}-learnls-manifold-nu2-K1300-stride4-learnls-bs1.0-bd0.01-bw0.1-knn15-faiss_atlas_weighted-5-randomwalk-ind1000-lr0.05-ep20-lbs10-blend0.8-infl50-rootdissolve-dn3-prune0.9510",
    "spectral (fixed Matern shape)":   f"FOLD-{FOLD}-spectral-nu2-K1300-stride4-bw0.1-knn15-faiss_atlas_weighted-5-randomwalk-lr0.05-ep20-lbs10-rootdissolve-dn310",
    "spectral (free per-mode weights)": f"FOLD-{FOLD}-spectral-nu2-K1300-stride4-bw0.1-knn15-faiss_atlas_weighted-5-randomwalk-lr0.005-ep20-lbs10-learndiff-learnspecw10",
}

def gt(raw, lb):
    return F.softplus(raw) + lb

SD = {}
for name, d in DEPLOYED_RUNS.items():
    ckpt = torch.load(PER_LIPID_ROOT / d / "checkpoints" / "batch_000.pt", map_location="cpu", weights_only=False)
    SD[name] = ckpt["model_state"]

lipids5 = ['PC 35:1 PE 38:1', 'PA 36:1', 'LPC 22:6', 'PE O-36:0 PE O-38:3', 'Hex2Cer 40:1;O2']

print("=== LEARNED HYPERPARAMETERS (read directly off the checkpoints) ===\n")
sd = SD["euclidean (ARD)"]
ls_euc_all = gt(sd["covar_module.base_kernel.raw_lengthscale"],
                sd["covar_module.base_kernel.raw_lengthscale_constraint.lower_bound"]).squeeze(1)
os_euc = gt(sd["covar_module.raw_outputscale"], sd["covar_module.raw_outputscale_constraint.lower_bound"])
print("EUCLIDEAN (nu=1.5, ARD -- one lengthscale per axis, per lipid):")
for l, row, o in zip(lipids5, ls_euc_all, os_euc):
    print(f"   {l:<22} ell=(x={row[0]:.3f}, y={row[1]:.3f}, z={row[2]:.3f})mm   outputscale={o:.4f}")

sd = SD["manifold (learned ell)"]
ell_man = gt(sd["covar_module.base_kernel.base_kernel.raw_lengthscale"],
             sd["covar_module.base_kernel.base_kernel.raw_lengthscale_constraint.lower_bound"]).item()
print(f"\nMANIFOLD (nu=2, K=1300, atlas x50 + prune 0.95): ell={ell_man:.3f}mm")

sd = SD["spectral (fixed Matern shape)"]
ell_spf = gt(sd["kernel.raw_lengthscale"], sd["kernel.raw_lengthscale_constraint.lower_bound"]).item()
print(f"SPECTRAL, fixed Matern S(lambda) (nu=2, K=1300, atlas x50 unpruned): ell={ell_spf:.3f}mm")

sd = SD["spectral (free per-mode weights)"]
ell_free = gt(sd["kernel.raw_lengthscale"], sd["kernel.raw_lengthscale_constraint.lower_bound"]).item()
w_free = gt(sd["kernel.raw_spectral_weights"], sd["kernel.raw_spectral_weights_constraint.lower_bound"]).squeeze(0).numpy()
print(f"SPECTRAL, FREE per-mode weights (nu=2, K=1300): own ell parameter={ell_free:.3f}mm (vestigial --")
print(f"   the {len(w_free)} free weights w_k replace S(lambda) entirely)")
print(f"   w_k: min={w_free.min():.4f}  max={w_free.max():.4f}  mean={w_free.mean():.4f}  "
      f"max/min = {w_free.max()/w_free.min():.0f}x")

print("\n=== HELD-OUT TEST CORRELATION, same 5 lipids ===")
for name, d in DEPLOYED_RUNS.items():
    df = pd.read_csv(PER_LIPID_ROOT / d / "metrics.csv").set_index("lipid_name")
    print(f"\n{name}:")
    for l in lipids5:
        print(f"   {l:<22} test_corr={df.loc[l,'test_corr']:.4f}   test_r2={df.loc[l,'test_r2']:.4f}")

**The deployed Euclidean kernel is far sharper than anything tested in Part 1.** ARD gives it a
lengthscale of only **~0.35-0.44mm per axis, per lipid** — roughly a third of Part 1's own
cross-validated $\ell\approx1.2$mm. Real marginal-likelihood training on the actual data, with a
learned noise term and per-lipid tuning, converged somewhere Part 1's simplified closed-form check
never tried. The manifold kernel's real learned $\ell=2.95$mm and the fixed-shape spectral kernel's
$\ell=2.15$mm both sit comfortably in the smooth, non-ringing zone from the Gibbs section above.

The free-weights kernel is the odd one out: its own $\ell$ parameter (0.69mm) is vestigial once 1,300
free weights are doing the real work, and those weights span **four orders of magnitude**
(0.006 to 58.8) — nothing like a smooth, gently-decaying $S(\lambda)$.

## Distance, covariogram, boundaries, Gibbs — for the real kernels

Same four checks as Part 1, now on the real $(\nu,\ell)$ values above, for a representative lipid
(`Hex2Cer 40:1;O2`, the best-predicted of the five). Euclidean uses its real ARD lengthscale; manifold
and spectral use the eigenbases above (exact for manifold, closest-available for spectral).

In [ ]:
from scipy.spatial.distance import cdist

EIGDIR = Path("/home/casap/mlibra/output/eigenvectors/eigvecs")
manifold_file = EIGDIR / ("bw=0.1_graph=bbox=None_denoise=3_k=15_method=faiss_atlas_weighted_nlist=729_"
                          "prune=0.95_stride=4_template=reference_thresh=5_weighting=atlas_x50_"
                          "rootdissolve_modes=1300_norm=randomwalk.eigpairs.npz")
spectral_file = EIGDIR / ("bw=0.1_graph=bbox=None_k=15_method=faiss_atlas_weighted_nlist=729_stride=4_"
                          "template=reference_thresh=5_weighting=atlas_x50_rootdissolve_modes=1300_"
                          "norm=randomwalk.eigpairs.npz")

fm = np.load(manifold_file); lam_m, vec_m = fm["eigval"].astype(np.float64), fm["eigvec"][anch_node_tr].astype(np.float64)
fs = np.load(spectral_file); lam_s, vec_s = fs["eigval"].astype(np.float64), fs["eigvec"][anch_node_tr].astype(np.float64)
print(f"manifold eigenbasis: {vec_m.shape}, spectral eigenbasis: {vec_s.shape}")

li = lipids5.index("Hex2Cer 40:1;O2")
ell_xyz_li = ls_euc_all[li].numpy()

I4, J4 = np.triu_indices(NTR, 1)
dd4_full = np.linalg.norm(anch_mm_tr[I4] - anch_mm_tr[J4], axis=1)
sel4 = dd4_full <= 2.0
I4, J4, dd4 = I4[sel4], J4[sel4], dd4_full[sel4]
XR4 = REGION_tr[I4] != REGION_tr[J4]

Xw = anch_mm_tr / ell_xyz_li
rho_euc_real = (lambda d, nu=1.5: (2**(1-nu)/gamma_fn(nu)) * (np.sqrt(2*nu)*np.maximum(d,1e-12))**nu
                * kv(nu, np.sqrt(2*nu)*np.maximum(d,1e-12)))(cdist(Xw, Xw)[I4, J4])

def rho_manifold_shape(lam, vec, ls, nu):
    Sd = (2 * nu / ls ** 2 + lam) ** (-nu); Sd /= Sd.sum()
    diag = ((vec * Sd) * vec).sum(1)
    return ((vec[I4] * Sd) * vec[J4]).sum(1) / np.sqrt(diag[I4] * diag[J4])

def rho_free_weights(lam, vec, w):
    w = w / w.sum()
    diag = ((vec * w) * vec).sum(1)
    return ((vec[I4] * w) * vec[J4]).sum(1) / np.sqrt(diag[I4] * diag[J4])

rho_man_real = rho_manifold_shape(lam_m, vec_m, ell_man, 2.0)
rho_spf_real = rho_manifold_shape(lam_s, vec_s, ell_spf, 2.0)
rho_free_real = rho_free_weights(lam_s, vec_s, w_free)
DEPLOYED_RHO = {"euclidean (ARD)": rho_euc_real, "manifold": rho_man_real,
                "spectral (fixed)": rho_spf_real, "spectral (free)": rho_free_real}

print(f"\n{'d (mm)':<10}{'euclidean':>11}{'manifold':>10}{'spec.fixed':>12}{'spec.free':>11}{'free frac<0':>13}")
for b in range(len(EDG) - 1):
    m = (dd4 >= EDG[b]) & (dd4 < EDG[b + 1])
    print(f"{EDG[b]}-{EDG[b+1]:<5}{rho_euc_real[m].mean():>11.3f}{rho_man_real[m].mean():>10.3f}"
          f"{rho_spf_real[m].mean():>12.3f}{rho_free_real[m].mean():>11.3f}"
          f"{100*(rho_free_real[m] < 0).mean():>12.0f}%")

print(f"\n{'d (mm)':<10}{'euclidean %':>13}{'manifold %':>12}{'spec.fixed %':>14}{'spec.free %':>13}   (boundary drop)")
for b in range(len(EDG) - 1):
    m = (dd4 >= EDG[b]) & (dd4 < EDG[b + 1])
    row = []
    for v in DEPLOYED_RHO.values():
        s_, c_ = v[m & ~XR4].mean(), v[m & XR4].mean()
        row.append(100 * (s_ - c_) / s_)
    print(f"{EDG[b]}-{EDG[b+1]:<5}{row[0]:>13.1f}{row[1]:>12.1f}{row[2]:>14.1f}{row[3]:>13.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4))
ctr4 = .5 * (EDG[1:] + EDG[:-1])
CC4 = {"euclidean (ARD)": COL["euclidean"], "manifold": COL["laplacian · atlas x50"],
       "spectral (fixed)": "#1baf7a", "spectral (free)": "#e34948"}

ax = axes[0]
S0_real = np.nanmean(Z_tr[I4] * Z_tr[J4], axis=1)
emp = [S0_real[(dd4>=EDG[i])&(dd4<EDG[i+1])].mean() for i in range(len(EDG)-1)]
emp_scale = emp[0]
ax.plot(ctr4, np.array(emp)/emp_scale, color="#111", lw=2.6, ls=":", label="empirical (normalised)")
for nm, v in DEPLOYED_RHO.items():
    row = np.array([v[(dd4>=EDG[i])&(dd4<EDG[i+1])].mean() for i in range(len(EDG)-1)])
    ax.plot(ctr4, row, color=CC4[nm], lw=2, marker="o", ms=4, label=nm)
ax.set_xlabel("distance (mm)"); ax.set_ylabel("implied correlation")
ax.set_title("(a) Distance decay, real learned hyperparameters")
ax.legend(fontsize=7.5)

ax = axes[1]
for nm, v in DEPLOYED_RHO.items():
    row = []
    for i in range(len(EDG)-1):
        m = (dd4>=EDG[i])&(dd4<EDG[i+1])
        s_, c_ = v[m & ~XR4].mean(), v[m & XR4].mean()
        row.append(100*(s_-c_)/s_)
    ax.plot(ctr4, row, color=CC4[nm], lw=2, marker="o", ms=4, label=nm)
ax.axhline(0, color="#999", lw=.8, ls=":")
ax.set_xlabel("distance (mm)"); ax.set_ylabel("border drop (%)")
ax.set_title("(b) Boundary drop, real learned hyperparameters")
ax.legend(fontsize=7.5)

ax = axes[2]
for nm, v in DEPLOYED_RHO.items():
    frac = [100*(v[(dd4>=EDG[i])&(dd4<EDG[i+1])] < 0).mean() for i in range(len(EDG)-1)]
    ax.plot(ctr4, frac, color=CC4[nm], lw=2, marker="o", ms=4, label=nm)
ax.set_xlabel("distance (mm)"); ax.set_ylabel("% of pairs with negative implied correlation")
ax.set_title("(c) Gibbs check: only free-weights rings")
ax.legend(fontsize=7.5)

fig.suptitle("Real deployed kernels: distance, boundary, and ringing — Hex2Cer 40:1;O2",
             fontsize=12, y=1.04)
fig.tight_layout()
plt.show()

**Distance & boundary.** All three properly-tapered kernels (Euclidean-ARD, manifold, spectral-fixed)
decay smoothly and never go negative — Euclidean fastest (its ARD lengthscale is far shorter),
manifold and spectral-fixed both retain a longer tail, consistent with their ~2-3mm learned $\ell$.
The manifold kernel's boundary drop (using its *exact* pruned graph) is far larger than the two
spectral variants (using the closest-available *unpruned* graph) — expected, since pruning
specifically removes cross-region edges and mechanically amplifies the border signal in the diffusion
embedding; this is a real difference in what each model was actually trained on, not an artifact of
the approximation.

**Gibbs.** Only **spectral (free)** ever produces a negative implied correlation, and it gets *worse*
with distance — 0% under 1mm, climbing to 7% by 1.6-2mm. The two kernels using a smooth, monotone
Matérn $S(\lambda)$ (manifold, spectral-fixed) never ring, at any distance tested, matching the
predictions from the Gibbs section above: a proper taper suppresses ringing; free weights are not
required to be a proper taper.

## The weakness of free per-mode weights

A monotone $S(\lambda)$ has exactly 2 free numbers ($\nu,\ell$) and is *forced* to be a smooth,
ranked function of eigenvalue. Free per-mode weights remove that constraint entirely — 1,300 numbers,
one per mode, no requirement that they decrease with $\lambda$ at all. Check whether the learned
weights actually behave like a sensible spectral density, or not.

In [ ]:
from scipy.stats import spearmanr

w_free_n = w_free / w_free.sum()
eff_free = 1.0 / (w_free_n ** 2).sum()

# what a SMOOTH Matern taper at this model's own (vestigial) ell would look like on the same eigenbasis
Sd_equiv = (2 * 2.0 / ell_free ** 2 + lam_s) ** (-2.0); Sd_equiv /= Sd_equiv.sum()
eff_equiv = 1.0 / (Sd_equiv ** 2).sum()

rho_mono = spearmanr(w_free, lam_s).statistic
print(f"effective modes -- free weights (actual):        {eff_free:.1f}")
print(f"effective modes -- smooth Matern at its OWN ell={ell_free:.2f}mm (what it 'should' look like): {eff_equiv:.1f}")
print(f"\nspearman(w_k, lambda_k): {rho_mono:.3f}  "
      f"(a real S(lambda) is monotone decreasing -> exactly -1.0; this is far from it)")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax = axes[0]
order = np.argsort(lam_s)
ax.plot(np.arange(1, len(lam_s)+1), w_free[order], color="#e34948", lw=.8, alpha=.8, label="learned $w_k$ (free)")
ax.plot(np.arange(1, len(lam_s)+1), Sd_equiv[order] * w_free.sum(), color="#111", lw=2, ls="--",
        label=f"smooth Matern at its own ell={ell_free:.2f}mm\n(same total mass, for scale)")
ax.set_yscale("log")
ax.set_xlabel("mode index (ascending eigenvalue)"); ax.set_ylabel("weight (log scale)")
ax.set_title("(a) The learned weights are not a smooth function of mode index")
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(lam_s, w_free, s=4, alpha=.35, color="#e34948")
ax.set_yscale("log")
ax.set_xlabel("eigenvalue $\\lambda_k$"); ax.set_ylabel("weight $w_k$ (log scale)")
ax.set_title(f"(b) vs eigenvalue directly (spearman={rho_mono:.2f}, a real S(λ) would be -1.0)")

fig.suptitle("Free per-mode weights: erratic, not a spectral density", fontsize=12, y=1.03)
fig.tight_layout()
plt.show()

print("\nheld-out test correlation, spectral (fixed shape) vs spectral (free weights), same 5 lipids:")
df_fixed = pd.read_csv(PER_LIPID_ROOT / DEPLOYED_RUNS["spectral (fixed Matern shape)"] / "metrics.csv").set_index("lipid_name")
df_free  = pd.read_csv(PER_LIPID_ROOT / DEPLOYED_RUNS["spectral (free per-mode weights)"] / "metrics.csv").set_index("lipid_name")
for l in lipids5:
    a, b = df_fixed.loc[l, "test_corr"], df_free.loc[l, "test_corr"]
    print(f"   {l:<22} fixed={a:.4f}   free={b:.4f}   fixed wins: {a > b}")

**Free per-mode weights do not converge to a sensible spectral density — they converge to noise with
structure in the wrong place.** Spearman correlation between the learned weight and its eigenvalue is
only **-0.37** (a real Matérn-type $S(\lambda)$ would be exactly $-1$): many high-frequency, rough
modes get *more* weight than some low-frequency, smooth ones, in flat violation of what any physically
sensible covariance would do. The effective mode count (~75) sits far below what the model's own
vestigial $\ell$ implies a rough process should want (~1,220 — nearly all 1,300 modes evenly) — instead
of spreading broadly and smoothly (avoiding ringing) or concentrating cleanly on the smoothest ordered
modes (like manifold/spectral-fixed do), it concentrates on a scattered, non-contiguous subset of ~75
modes with no respect for their ordering. That is the mechanism behind the Gibbs-style ringing measured
above: a smooth taper is what suppresses ringing, and an erratic weight vector is not a taper at all,
regardless of how few effective modes it ends up using.

**And it costs real accuracy.** On every one of the 5 lipids, the fixed-Matérn-shape spectral kernel
beats the free-per-mode-weights version on held-out test correlation — a small margin
(~0.005-0.012), but the same direction on all five, matching everything else in this notebook:
more flexibility in the wrong place (unconstrained per-mode weights instead of a physically-motivated
taper) makes the kernel *worse*, not better. Constraining the shape was never the limitation; it was
protecting the model from exactly this failure mode.